In [1]:
!pip install /kaggle/input/pip-install-lifelines/autograd-1.7.0-py3-none-any.whl
!pip install /kaggle/input/pip-install-lifelines/autograd-gamma-0.5.0.tar.gz
!pip install /kaggle/input/pip-install-lifelines/interface_meta-1.3.0-py3-none-any.whl
!pip install /kaggle/input/pip-install-lifelines/formulaic-1.0.2-py3-none-any.whl
!pip install /kaggle/input/pip-install-lifelines/lifelines-0.30.0-py3-none-any.whl

Processing /kaggle/input/pip-install-lifelines/autograd-1.7.0-py3-none-any.whl
autograd is already installed with the same version as the provided wheel. Use --force-reinstall to force an installation of the wheel.
Processing /kaggle/input/pip-install-lifelines/autograd-gamma-0.5.0.tar.gz
  Preparing metadata (setup.py) ... done
  Created wheel for autograd-gamma: filename=autograd_gamma-0.5.0-py3-none-any.whl size=4031 sha256=435d4a79f281f7ba89fe5fcb543a21e8a457c8bf7092f9d1c017b1f9f18d7ca8
  Stored in directory: /root/.cache/pip/wheels/6b/b5/e0/4c79e15c0b5f2c15ecf613c720bb20daab20a666eb67135155
Successfully built autograd-gamma
Processing /kaggle/input/pip-install-lifelines/interface_meta-1.3.0-py3-none-any.whl
Processing /kaggle/input/pip-install-lifelines/formulaic-1.0.2-py3-none-any.whl
Processing /kaggle/input/pip-install-lifelines/lifelines-0.30.0-py3-none-any.whl


In [2]:
import warnings
warnings.simplefilter(action='ignore')

import os
import requests
import joblib

import numpy
import pandas

import matplotlib.pyplot as plt
import scipy.stats as stats
import tensorflow_decision_forests as tfdf

In [3]:
from catboost import CatBoostRegressor ,Pool
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.linear_model import ElasticNetCV, LassoCV, RidgeCV
from sklearn.svm import SVR

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.ensemble import AdaBoostRegressor

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import accuracy_score

from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_val_score

from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif
from sklearn.feature_selection import SelectFromModel

from sklearn.impute import SimpleImputer
import sklearn.linear_model as linear_model
from sklearn.model_selection import train_test_split
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

In [4]:
train = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/train.csv').drop(columns=['ID'])
test = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/test.csv').drop(columns=['ID'])

In [5]:
from lifelines import KaplanMeierFitter

def calculate_survival_probabilities(df, time_col, event_col):
    kmf = KaplanMeierFitter()
    kmf.fit(df[time_col], df[event_col])
    return kmf.survival_function_at_times(df[time_col]).values

def preprocess_survival_data(df, time_col='efs_time', event_col='efs'):
    df['target'] = calculate_survival_probabilities(df, time_col, event_col)
    df.loc[df[event_col] == 0, 'target'] -= 0.25  # Adjust for censored data
    return df

train = preprocess_survival_data(train)

In [6]:
encoder = LabelEncoder()
normalize = SimpleImputer(strategy='mean')

In [7]:
for i in zip(train.columns,train.dtypes):
    if i[1]=='O':
        train[i[0]] = train[i[0]].fillna('Unknown')
        train[i[0]]=normalize.fit_transform(encoder.fit_transform(train[i[0]].to_numpy().reshape(-1,1)).reshape(-1,1))
    else:
        train[i[0]] = train[i[0]].fillna(0)
        train[i[0]]=normalize.fit_transform(numpy.array(train[i[0]]).reshape(-1,1))
for i in zip(test.columns,test.dtypes):
    if i[1]=='O':
        test[i[0]] = test[i[0]].fillna('Unknown')
        test[i[0]]=normalize.fit_transform(numpy.array(encoder.fit_transform(test[i[0]].to_numpy().reshape(-1,1))).reshape(-1,1))
    else:
        test[i[0]] = test[i[0]].fillna(0)
        test[i[0]]=normalize.fit_transform(numpy.array(test[i[0]]).reshape(-1,1))

In [8]:
train_x = train.drop(columns=['efs', 'efs_time', 'target'])
train_y = train['target']
    
#train.hist(figsize=(16, 20), bins=50, xlabelsize=8, ylabelsize=8)

#clf = GradientBoostingRegressor(n_estimators=100).fit(train_x, train_y)
#model = SelectFromModel(clf, prefit=True)

#weight = model.transform(train_x).reshape(-1)
train.head()

,dri_score,psych_disturb,cyto_score,diabetes,hla_match_c_high,hla_high_res_8,tbi_status,arrhythmia,hla_low_res_6,graft_type,...,donor_related,melphalan_dose,hla_low_res_8,cardiac,hla_match_drb1_high,pulm_moderate,hla_low_res_10,efs,efs_time,target
0,7.0,0.0,7.0,0.0,0.0,0.0,0.0,0.0,6.0,0.0,...,3.0,1.0,8.0,0.0,2.0,0.0,10.0,0.0,42.356,0.208687
1,2.0,0.0,1.0,0.0,2.0,8.0,6.0,0.0,6.0,1.0,...,1.0,1.0,8.0,0.0,2.0,3.0,10.0,1.0,4.672,0.847759
2,7.0,0.0,7.0,0.0,2.0,8.0,0.0,0.0,6.0,0.0,...,1.0,1.0,8.0,0.0,2.0,0.0,10.0,0.0,19.793,0.212424
3,0.0,0.0,1.0,0.0,2.0,8.0,0.0,0.0,6.0,0.0,...,3.0,1.0,8.0,0.0,2.0,0.0,10.0,0.0,102.349,0.206661
4,0.0,0.0,7.0,0.0,2.0,8.0,0.0,0.0,6.0,1.0,...,1.0,0.0,8.0,0.0,2.0,0.0,10.0,0.0,16.223,0.214674


In [9]:
Fold=5
kf = KFold(n_splits=Fold)

LGBM_pre_train = numpy.zeros(len(train))
LGBM_pre_test = numpy.zeros(len(test))

xgb_pre_train = numpy.zeros(len(train))
xgb_pre_test = numpy.zeros(len(test))

cat_pre_train = numpy.zeros(len(train))
cat_pre_test = numpy.zeros(len(test))

RandomF_pre_train = numpy.zeros(len(train))
RandomF_pre_test = numpy.zeros(len(test))

for i, (train_index, test_index) in enumerate(kf.split(train)):

    print(f"FOLD: {i}")

    x_fold=train_x[train_index[0]: train_index[len(train_index)-1]]
    y_fold=train_y[train_index[0]: train_index[len(train_index)-1]]

    x_test_fold=train_x[test_index[0]: test_index[len(test_index)-1]]
    y_test_fold=train_y[test_index[0]: test_index[len(test_index)-1]]

    #weight_ = weight[train_index[0]: train_index[len(train_index)-1]]

    lightgbm = LGBMRegressor()
    xgboost = XGBRegressor()
    catboost = CatBoostRegressor(learning_rate=0.009,
                                 depth=5,
                                 l2_leaf_reg=3.5,
                                 min_child_samples=32,
                                 grow_policy='Depthwise',
                                 iterations=1000,
                                 eval_metric='RMSE',
                                 od_type='Iter',
                                 od_wait=50,
                                 random_state=42,
                                 logging_level='Silent')
    RandomF = GradientBoostingRegressor()
    
    lightgbm.fit(x_fold, y_fold)#, eval_sample_weight=weight_)
    xgboost.fit(x_fold, y_fold)#, sample_weight=weight_)
    catboost.fit(x_fold, y_fold)#, sample_weight=weight_)
    RandomF.fit(x_fold, y_fold)#, weight_)

    LGBM_pre_train+=lightgbm.predict(train_x)
    LGBM_pre_test+=lightgbm.predict(test)

    xgb_pre_train+=xgboost.predict(train_x)
    xgb_pre_test+=xgboost.predict(test)

    cat_pre_train+=catboost.predict(train_x)
    cat_pre_test+=catboost.predict(test)

    RandomF_pre_train += RandomF.predict(train_x)
    RandomF_pre_test += RandomF.predict(test)

    print(RandomF.score(x_test_fold, y_test_fold))
    print(catboost.score(x_test_fold, y_test_fold))
    print(xgboost.score(x_test_fold, y_test_fold))
    print(lightgbm.score(x_test_fold, y_test_fold))

FOLD: 0
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.008452 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 811
[LightGBM] [Info] Number of data points in the train set: 23039, number of used features: 57
[LightGBM] [Info] Start training from score 0.491454
0.18513039796741348
0.19191952688286018
0.1447232711743066
0.19388061675835544
FOLD: 1
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006899 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 811
[LightGBM] [Info] Number of data points in the train set: 28799, number of used features: 57
[LightGBM] [Info] Start training from score 0.491025
0.19186371622632736
0.22077146557934957
0.5347166274480927
0.301402313685603
FOLD: 2
[LightGBM]

new_train_data = pandas.DataFrame({'RandomF' : RandomF_pre_train,
                             'LGBM' : LGBM_pre_train,
                             'xgb' : xgb_pre_train,
                             'cat' : cat_pre_train})

new_test_data = pandas.DataFrame({'RandomF' : RandomF_pre_test,
                             'LGBM' : LGBM_pre_test,
                             'xgb' : xgb_pre_test,
                             'cat' : cat_pre_test})
model=GradientBoostingRegressor()
model.fit(new_train_data, train_y)

In [10]:
id = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/test.csv')['ID']
test_predictions = ((RandomF_pre_test/Fold)+(LGBM_pre_test/Fold)+(xgb_pre_test/Fold)+(cat_pre_test/Fold))/4
#((RandomF_pre_test/Fold)+(LGBM_pre_test/Fold)+(xgb_pre_test/Fold)+(cat_pre_test/Fold))/4
#model.predict(new_test_data)
#((RandomF_pre_test/10)+(LGBM_pre_test/5)+(xgb_pre_test/5)+(cat_pre_test/5))/4
test_predictions

array([0.29906621, 0.73459585, 0.33957838])

In [11]:
submission = pandas.DataFrame({
    'ID': id.values,
    'prediction': test_predictions,
})
submission

,ID,prediction
0,28800,0.299066
1,28801,0.734596
2,28802,0.339578


In [12]:
submission.to_csv('submission.csv', index = False)